# Diagnostic 1: unique-image-level reassignment (CLIP BASED)


In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [ ]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.diagnostic_dataloader import load_multimodal_tensors_shuffled_image
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+clip (diagnostic 1)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'clip'
IMG_FEAT_DIM = 768

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '1_diagnostic'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
STABILITY_SUMMARY_PATH = (
    RESULTS_DIR / 'diagnostic1_unique_image_reassignment_clip_validation_summary.json'
)


In [ ]:
stats = data.load_cgm_stats()

_preview_loader = load_multimodal_tensors_shuffled_image(
    'train', encoder_name=ENCODER_NAME, seed=SEEDS[0]
)
_sample_batch = next(iter(_preview_loader))

print('batch shapes:', [tuple(x.shape) for x in _sample_batch])
print('Diagnostic 1 uses train + validation only; the held-out test set is not loaded.')


batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]
Diagnostic 1 uses train + validation only; the held-out test set is not loaded.


In [ ]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    # One fixed unique-image reassignment per seed/split.
    train_loader = load_multimodal_tensors_shuffled_image(
        'train', encoder_name=ENCODER_NAME, seed=seed
    )
    val_loader = load_multimodal_tensors_shuffled_image(
        'val', encoder_name=ENCODER_NAME, seed=seed, shuffle=False
    )

    train.set_seed(seed)
    model = MultimodalGRUPredictor(
        img_feat_dim=IMG_FEAT_DIM,
        gru_hidden_size=256,
        gru_num_layers=3,
        gru_dropout=0.2,
    )

    model, history = train.train_model(
        model,
        train_loader,
        val_loader,
        stats['cgm_mean'],
        stats['cgm_std'],
        lr=0.0001,
        weight_decay=1e-05,
        forward_fn=forward_fn,
        checkpoint_dir=(
            EXPERIMENT_DIR / 'checkpoints'
            / f'diagnostic1_unique_image_reassignment_clip_seed{seed}'
        ),
        verbose=False,
        progress_bar=True,
        progress_desc=f'seed={seed}',
    )

    n_epochs = len(history['train_loss'])
    val_history = history.get('val_loss', [])
    best_epoch = int(np.argmin(val_history) + 1) if len(val_history) else n_epochs

    y_pred_val, y_true_val = train.predict(
        model, val_loader, forward_fn=forward_fn
    )
    val_results = metrics.evaluate(
        y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std']
    )

    seed_result = {
        'model': MODEL_NAME,
        'seed': seed,
        'n_epochs_trained': n_epochs,
        'best_epoch': best_epoch,
        'train_loss_history': [float(x) for x in history['train_loss']],
        'val_loss_history': [float(x) for x in val_history],
        'val_results': val_results,
    }
    all_results.append(seed_result)

    with open(
        RESULTS_DIR / f'diagnostic1_unique_image_reassignment_clip_seed{seed}_results.json',
        'w',
    ) as f:
        json.dump(seed_result, f, indent=2)

    val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
    print(
        f"seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  "
        f"best epoch={best_epoch:>3}  "
        f"validation RMSE @ {HORIZONS} min = {val_rmse}\n"
    )


Validation results — CGM+time+clip (diagnostic 1) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.39, 17.47, 25.16, 30.05, 33.16]

Validation results — CGM+time+clip (diagnostic 1) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  15/50  best epoch=  5  validation RMSE @ [15, 30, 60, 90, 120] min = [10.9, 17.39, 25.29, 29.92, 32.83]

Validation results — CGM+time+clip (diagnostic 1) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


...

seed= 56  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.52, 17.66, 25.42, 30.19, 33.22]

Validation results — CGM+time+clip (diagnostic 1) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  14/50  best epoch=  4  validation RMSE @ [15, 30, 60, 90, 120] min = [11.11, 17.49, 25.52, 30.39, 33.33]

Validation results — CGM+time+clip (diagnostic 1) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.51, 17.68, 25.33, 30.12, 33.19]



In [ ]:
# Diagnostic summary: validation only.
# The held-out test set remains untouched during this model-development diagnostic.

METRICS = ['rmse', 'mae', 'grmse']
summary = {}

for h in HORIZONS:
    summary[h] = {}
    for metric_name in METRICS:
        values = [r['val_results'][h][metric_name] for r in all_results]
        summary[h][metric_name] = {
            'mean': float(np.mean(values)),
            'sd_across_seeds': float(np.std(values, ddof=1)),
        }

best_epochs = [r['best_epoch'] for r in all_results]
stopping_epochs = [r['n_epochs_trained'] for r in all_results]

final_loss_gaps = []
for r in all_results:
    if r['val_loss_history']:
        final_loss_gaps.append(
            float(r['val_loss_history'][-1] - r['train_loss_history'][-1])
        )

print(f'=== {MODEL_NAME}: validation summary across seeds ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min: "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd_across_seeds']:.2f}   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd_across_seeds']:.2f}   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd_across_seeds']:.2f}"
    )

print(
    f"Best epoch: {np.mean(best_epochs):.2f} +/- {np.std(best_epochs, ddof=1):.2f}; "
    f"stopping epoch: {np.mean(stopping_epochs):.2f} +/- "
    f"{np.std(stopping_epochs, ddof=1):.2f}"
)

if final_loss_gaps:
    print(
        f"Final normalized val-train loss gap: "
        f"{np.mean(final_loss_gaps):.6f} +/- {np.std(final_loss_gaps, ddof=1):.6f}"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump(
        {
            'model': MODEL_NAME,
            'seeds': SEEDS,
            'n_seeds': len(SEEDS),
            'per_horizon_validation_summary': summary,
            'best_epochs': best_epochs,
            'stopping_epochs': stopping_epochs,
            'final_normalized_val_minus_train_loss_gaps': final_loss_gaps,
            'test_set_evaluated': False,
        },
        f,
        indent=2,
    )

print(f'\nsaved per-seed validation results to {RESULTS_DIR}')
print(f'saved validation summary to {STABILITY_SUMMARY_PATH}')


=== CGM+time+clip (diagnostic 1): validation summary across seeds ===
 15-min: RMSE = 11.27 +/- 0.23   MAE = 7.71 +/- 0.17   gRMSE = 12.31 +/- 0.27
 30-min: RMSE = 17.53 +/- 0.10   MAE = 11.82 +/- 0.10   gRMSE = 19.77 +/- 0.26
 60-min: RMSE = 25.34 +/- 0.12   MAE = 16.83 +/- 0.14   gRMSE = 30.02 +/- 0.27
 90-min: RMSE = 30.16 +/- 0.14   MAE = 20.18 +/- 0.18   gRMSE = 36.51 +/- 0.29
120-min: RMSE = 33.18 +/- 0.16   MAE = 22.63 +/- 0.20   gRMSE = 40.49 +/- 0.30
Best epoch: 3.00 +/- 1.15; stopping epoch: 13.00 +/- 1.15
Final normalized val-train loss gap: 0.072852 +/- 0.007708

saved per-seed validation results to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/1_diagnostic/results
saved validation summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/1_diagnostic/results/diagnostic1_unique_image_reassignment_clip_validation_summary.json


### Comparation with real images clip

In [ ]:
REAL_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' /'multimodal' /'results' /'gated_clip'
REAL_STUB = 'gated_cgm_time_clip'
DIAG_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '1_diagnostic' /'results'
DIAG_STUB = 'diagnostic1_unique_image_reassignment_clip'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
HORIZONS = [15, 30, 60, 90, 120]
EARLY_STOP_PATIENCE = 10

In [ ]:
def load_seed_results(dir_, stub, seeds):
    out = []
    for seed in seeds:
        path = dir_ / f'{stub}_seed{seed}_results.json'
        if not path.exists():
            raise FileNotFoundError(f'missing {path} -- has this run finished all {len(seeds)} seeds?')
        with open(path) as f:
            out.append(json.load(f))
    return out

real_results = load_seed_results(REAL_DIR, REAL_STUB, SEEDS)
diag_results = load_seed_results(DIAG_DIR, DIAG_STUB, SEEDS)
print(f'loaded {len(real_results)} real-image seeds, {len(diag_results)} diagnostic1 seeds')

loaded 10 real-image seeds, 10 diagnostic1 seeds


In [ ]:
# Epoch-count comparison: how fast did each condition stop / (for diagnostic1) hit its best epoch
real_stopping = [r['n_epochs_trained'] for r in real_results]
real_best_inferred = [max(1, n - EARLY_STOP_PATIENCE) for n in real_stopping]
diag_stopping = [r['n_epochs_trained'] for r in diag_results]
diag_best = [r['best_epoch'] for r in diag_results]

print('=== Epoch counts (mean +/- sd across 10 seeds) ===')
print(f"{'':<28}{'stopping epoch':>18}{'best epoch':>18}")
print(f"{'real (correct image)':<28}"
      f"{np.mean(real_stopping):>12.2f}+/-{np.std(real_stopping, ddof=1):<4.2f}"
      f"{np.mean(real_best_inferred):>12.2f}+/-{np.std(real_best_inferred, ddof=1):<4.2f} (inferred)")
print(f"{'diagnostic1 (reassigned)':<28}"
      f"{np.mean(diag_stopping):>12.2f}+/-{np.std(diag_stopping, ddof=1):<4.2f}"
      f"{np.mean(diag_best):>12.2f}+/-{np.std(diag_best, ddof=1):<4.2f}")

diag_loss_gaps = []
for r in diag_results:
    if r.get('val_loss_history') and r.get('train_loss_history'):
        diag_loss_gaps.append(r['val_loss_history'][-1] - r['train_loss_history'][-1])
if diag_loss_gaps:
    print(f"\ndiagnostic1 final (val - train) loss gap: "
          f"{np.mean(diag_loss_gaps):.6f} +/- {np.std(diag_loss_gaps, ddof=1):.6f}")
print("real run's per-epoch loss history was not saved -- no directly-comparable loss-gap number "
      "(see markdown cell above)")

=== Epoch counts (mean +/- sd across 10 seeds) ===
                                stopping epoch        best epoch
real (correct image)               11.50+/-1.27        1.50+/-1.27 (inferred)
diagnostic1 (reassigned)           13.00+/-1.15        3.00+/-1.15

diagnostic1 final (val - train) loss gap: 0.072852 +/- 0.007708
real run's per-epoch loss history was not saved -- no directly-comparable loss-gap number (see markdown cell above)


In [ ]:
print('=== Validation RMSE: real (correct image) vs. diagnostic1 (reassigned image) ===')
print(f"{'horizon':>10}{'real':>14}{'diagnostic1':>16}{'delta':>10}")
for h in HORIZONS:
    real_vals = [r['val_results'][str(h)]['rmse'] for r in real_results]
    diag_vals = [r['val_results'][str(h)]['rmse'] for r in diag_results]
    real_mean, diag_mean = np.mean(real_vals), np.mean(diag_vals)
    print(f"{h:>8}min{real_mean:>10.2f}+/-{np.std(real_vals, ddof=1):<4.2f}"
          f"{diag_mean:>12.2f}+/-{np.std(diag_vals, ddof=1):<4.2f}"
          f"{diag_mean - real_mean:>+10.2f}")

print()
print('If diagnostic1 (reassigned image) closely matches real (correct image) on both epoch-count')
print('and RMSE -- the failure has nothing to do with image content, only with adding a branch of')
print('this shape/capacity. A clear gap in either direction means the true image-target')
print('correspondence itself matters.')

=== Validation RMSE: real (correct image) vs. diagnostic1 (reassigned image) ===
   horizon          real     diagnostic1     delta
      15min     12.23+/-0.50       11.27+/-0.23     -0.96
      30min     18.10+/-0.17       17.53+/-0.10     -0.57
      60min     25.75+/-0.18       25.34+/-0.12     -0.41
      90min     30.56+/-0.14       30.16+/-0.14     -0.40
     120min     33.33+/-0.09       33.18+/-0.16     -0.16

If diagnostic1 (reassigned image) closely matches real (correct image) on both epoch-count
and RMSE -- the failure has nothing to do with image content, only with adding a branch of
this shape/capacity. A clear gap in either direction means the true image-target
correspondence itself matters.


In [ ]:
from scipy import stats
import numpy as np

print('=== Paired comparison: correct image vs. reassigned image ===')
print(
    f"{'horizon':>10}"
    f"{'correct':>12}"
    f"{'reassigned':>14}"
    f"{'diff':>10}"
    f"{'95% CI':>22}"
    f"{'p-value':>12}"
)

paired_results = {}

for h in HORIZONS:
    correct = np.array([
        r['val_results'][str(h)]['rmse']
        for r in real_results
    ])

    reassigned = np.array([
        r['val_results'][str(h)]['rmse']
        for r in diag_results
    ])

    # Paired difference:
    # positive = correct image has higher (worse) RMSE
    diff = correct - reassigned

    mean_diff = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)
    se_diff = stats.sem(diff)

    # 95% CI for paired mean difference
    ci_low, ci_high = stats.t.interval(
        0.95,
        df=len(diff) - 1,
        loc=mean_diff,
        scale=se_diff,
    )

    # Two-sided paired t-test
    t_stat, p_value = stats.ttest_rel(correct, reassigned)

    paired_results[h] = {
        'mean_difference_correct_minus_reassigned': float(mean_diff),
        'sd_difference': float(sd_diff),
        'ci95': [float(ci_low), float(ci_high)],
        't_statistic': float(t_stat),
        'p_value': float(p_value),
    }

    print(
        f"{h:>8}min"
        f"{np.mean(correct):>12.2f}"
        f"{np.mean(reassigned):>14.2f}"
        f"{mean_diff:>+10.2f}"
        f"{f'[{ci_low:.2f}, {ci_high:.2f}]':>22}"
        f"{p_value:>12.4f}"
    )

=== Paired comparison: correct image vs. reassigned image ===
   horizon     correct    reassigned      diff                95% CI     p-value
      15min       12.23         11.27     +0.96          [0.61, 1.31]      0.0002
      30min       18.10         17.53     +0.57          [0.45, 0.70]      0.0000
      60min       25.75         25.34     +0.41          [0.27, 0.54]      0.0001
      90min       30.56         30.16     +0.40          [0.25, 0.55]      0.0002
     120min       33.33         33.18     +0.16          [0.05, 0.26]      0.0075
